In [ ]:
# RSNA Knee — offline inference (internet OFF). Weights: any attached *.pt export.
import glob, os, subprocess, sys, time

def walk_find(pred, root="/kaggle/input", max_depth=5):
    """Depth-limited search that never descends into the DICOM trees."""
    root = root.rstrip("/")
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath[len(root):].count(os.sep)
        dirnames[:] = [] if depth >= max_depth else [d for d in dirnames if not d.endswith("_series")]
        hit = pred(dirpath, filenames)
        if hit:
            return hit
    return None

CODE = walk_find(lambda d, f: d if os.path.isfile(os.path.join(d, "src", "models", "inference.py")) else None)
COMP = walk_find(lambda d, f: d if "test.csv" in f and os.path.isdir(os.path.join(d, "test_series")) else None)
assert CODE and COMP, (CODE, COMP)
sys.path.insert(0, CODE)
print("code:", CODE, "| competition:", COMP)


In [ ]:
WHEELS = walk_find(lambda d, f: d if os.path.basename(d) == "wheels" and any(x.endswith(".whl") for x in f) else None)
if WHEELS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", WHEELS,
                    "pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg"], check=False)
CKPTS = []
walk_find(lambda d, f: CKPTS.extend(os.path.join(d, x) for x in f if x.endswith(".pt")) and None)
print("wheels:", WHEELS); print("ckpts:", CKPTS)
assert CKPTS, "attach a weights dataset / training kernel output"

In [ ]:
t0 = time.time()
from src.models.inference import main
sys.argv = ["inference", "--data_dir", COMP, "--ckpts", *CKPTS, "--cache_dir", "/tmp/knee_cache",
            "--workers", str(os.cpu_count()), "--out", "/kaggle/working/submission.csv"]
main()
print(f"inference {time.time() - t0:.0f}s")

In [ ]:
import numpy as np, pandas as pd
sub = pd.read_csv("/kaggle/working/submission.csv")
files = glob.glob("/tmp/knee_cache/*/*.npz")
print(sub.shape, "| slot coverage on test:", np.mean([np.load(f)["valid"].mean() for f in files]).round(3))
sub.head()